In [ ]:
# Step 1: Install all required libraries for Retrieval, Generation, and individual Metrics
!pip install rank_bm25 nltk rouge-score evaluate bert-score transformers

In [ ]:
# Step 2: Define the KnowledgeRetriever and AnswerGenerator classes
def generate(self, context: str, query: str) -> str:
        # A more aggressive prompt engineering structure to keep GPT-2 on track
        prompt = (
            f"Fact: {context}\n"
            f"Question: {query}\n"
            f"Exact Answer based only on the fact:"
        )
        response = self.generator(
            prompt,
            max_new_tokens=15,  # Shortened so it stops right after giving the answer
            clean_up_tokenization_spaces=True,
            pad_token_id=50256,
            temperature=0.1,    # Low temperature makes the model less creative and more literal
            do_sample=True
        )
        full_text = response[0]['generated_text']
        # Extract the text right after our final prompt marker
        return full_text.split("Exact Answer based only on the fact:")[-1].strip()

In [ ]:
# Step 3: Run this cell to upload your custom text document (.txt)
from google.colab import files

print("Click 'Choose Files' below to upload your text document:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    file_content = uploaded[filename].decode("utf-8")
    retriever.load_document(file_content)
else:
    print("[Error] No file uploaded.")

In [ ]:
# Step 4: Submit your question to the Vectorless RAG system
query = input("Enter your question: ")

# Run the RAG pipeline
context = retriever.retrieve(query)
generated_answer = generator.generate(context, query)

print("\n" + "="*40)
print(f"Retrieved Context: {context}")
print(f"Generated Answer:  {generated_answer}")
print("="*40)
print("\nVariables saved! You can now run any of the metric cells below.")

In [ ]:
# Step 5: Evaluate the generated answer using the BLEU metric
import evaluate

ground_truth = input("Enter the Ground Truth (ideal answer) to evaluate against BLEU: ")

bleu_metric = evaluate.load("bleu")
result = bleu_metric.compute(predictions=[generated_answer], references=[ground_truth])

print(f"\n📈 BLEU Score: {result.get('bleu', 0.0):.4f}")

In [ ]:
# Step 6: Evaluate the generated answer using the ROUGE metric
import evaluate

ground_truth = input("Enter the Ground Truth (ideal answer) to evaluate against ROUGE: ")

rouge_metric = evaluate.load("rouge")
result = rouge_metric.compute(predictions=[generated_answer], references=[ground_truth])

print(f"\n📈 ROUGE-1: {result.get('rouge1', 0.0):.4f}")
print(f"📈 ROUGE-L: {result.get('rougeL', 0.0):.4f}")

In [ ]:
# Step 7: Evaluate the generated answer using the METEOR metric
import evaluate
import nltk
nltk.download('wordnet', quiet=True)

ground_truth = input("Enter the Ground Truth (ideal answer) to evaluate against METEOR: ")

meteor_metric = evaluate.load("meteor")
result = meteor_metric.compute(predictions=[generated_answer], references=[ground_truth])

print(f"\n📈 METEOR Score: {result.get('meteor', 0.0):.4f}")

In [ ]:
# Step 8: Evaluate the generated answer using the BERTScore metric
import evaluate

ground_truth = input("Enter the Ground Truth (ideal answer) to evaluate against BERTScore: ")

bertscore_metric = evaluate.load("bertscore")
result = bertscore_metric.compute(predictions=[generated_answer], references=[ground_truth], lang="en")

print(f"\n📈 BERTScore (F1): {result.get('f1', [0.0])[0]:.4f}")